# Real World Application

## Packages

In [ ]:
import pandas as pd
import numpy as np
import math
import torch
from matplotlib import pyplot as plt
import time

# GPytorch
import gpytorch
from gpytorch.means import ConstantMean
from gpytorch.kernels import ScaleKernel, MaternKernel, InducingPointKernel
from gpytorch.distributions import MultivariateNormal
from gpytorch.models import ApproximateGP
from gpytorch.variational import CholeskyVariationalDistribution
from gpytorch.variational import VariationalStrategy

# GPBoost
import gpboost as gpb

# DKL-GP
import viva
from viva import VIVACpp as VIVA, my_train_cpp as my_train

# Train-Test-Split
from sklearn.model_selection import train_test_split, KFold
from sklearn.metrics import roc_auc_score
# Batch
from torch.utils.data import TensorDataset, DataLoader

# Notebook
import tqdm
import tqdm.notebook as tqdm

# For CRPS & Log-Score
from scipy.stats import norm
from scipy.special import erf


from sklearn.cluster import KMeans

## Functions

In [ ]:
# Scale
def Scale(X_train, X_test, y_train, y_test):
    mins = X_train.min(0)
    maxs = X_train.max(0)
    X_train = (X_train-mins)/(maxs-mins)
    X_test = (X_test-mins)/(maxs-mins)

    #mean = y_train.mean()
    #sd = np.sqrt(y_train.var())

    #y_train = (y_train-mean)/sd
    #y_test = (y_test-mean)/sd
    
    return X_train, X_test, y_train, y_test

# Find too close points
def find_unique_X(X):
    n = len(X)
    mask = torch.ones(n, dtype=torch.bool)
    with torch.no_grad():
        for i in range(n - 1, 0, -1):
            if torch.cdist(X[i:i + 1], X[:i]).min() < 0.001:
                mask[i] = False
    return mask

# CRPS
def crps_norm_vectorized(observations, means, sigmas):
    """
    Compute CRPS for multiple normal distributions.
    
    Parameters:
    observations (array-like): Observed values
    means (array-like): Means of the forecast distributions
    sigmas (array-like): Variance of the forecast distributions
    
    Returns:
    array-like: CRPS values
    """
    observations = np.asarray(observations)
    means = np.asarray(means)
    sigmas = np.sqrt(np.asarray(sigmas))
    
    z = (observations - means) / sigmas
    crps = sigmas * (z * (2 * norm.cdf(z) - 1) + 2 * norm.pdf(z) - 1 / np.sqrt(np.pi))
    return np.mean(crps)

# Log-Score
def log_score_norm_vectorized(observations, means, sigmas):
    """
    Compute Log Score for multiple normal distributions.
    
    Parameters:
    observations (array-like): Observed values
    means (array-like): Means of the forecast distributions
    sigmas (array-like): Variance of the forecast distributions
    
    Returns:
    array-like: Log Score values
    """
    observations = np.asarray(observations)
    means = np.asarray(means)
    sigmas = np.sqrt(np.asarray(sigmas))
    
    return np.mean(-norm.logpdf(observations, means, sigmas))

## Data

Choose dataset:

In [ ]:
data = pd.read_csv('https://raw.githubusercontent.com/TimGyger/VIF/refs/heads/main/Real%20World/Non-Gaussian/Credit/credit.txt', sep=" ")

In [ ]:
data.head(3)

In [ ]:
# Features & Response
X = data.iloc[:, 1:]
y = data.iloc[:, 0]

X = X.values  # Selecting multiple features
y = y.values  # Selecting the target variable

mins = X.min(0)
maxs = X.max(0)
X1 = (X-mins)/(maxs-mins)
# Compute column-wise standard deviation
stds = np.std(X1, axis=0)

# Filter columns based on standard deviation
mask = stds > 0.1  # Keep columns with std > 0.1
X = X[:, mask]
mins = X.min(0)
maxs = X.max(0)
X1 = (X-mins)/(maxs-mins)
X1_tensor = torch.tensor(X1, dtype=torch.float32)
mask_all = find_unique_X(X1_tensor)
X = X1[mask_all]
y = y[mask_all]
d = X.shape[1] # Features
d

In [ ]:
# Generate 5 disjoint test sets
kf = KFold(n_splits=5, shuffle=True, random_state=100)
test_indices_sets = []
train_indices_sets = []
for i, (_, test_indices) in enumerate(kf.split(X)):
    print(f"Fold {i+1} Test indices: {test_indices}")
    test_indices_sets.append(set(test_indices))
    train_indices_sets.append(set(range(len(X))) - test_indices_sets[i])


## Models

### Sparse Gaussian Process Regression (SGPR)

#### Parameters

In [ ]:
num_inducing_points = 1000
training_iterations = 500

#### Model

In [ ]:
# SGPR
class SGPR(ApproximateGP):
    def __init__(self, train_x, num_inducing_points):
        inducing_points = train_x[:num_inducing_points, :].clone()
        variational_distribution = CholeskyVariationalDistribution(num_inducing_points)
        variational_strategy = VariationalStrategy(self, inducing_points, variational_distribution, learn_inducing_locations=True)
        super(SGPR, self).__init__(variational_strategy)
        
        self.mean_module = gpytorch.means.ConstantMean()
        self.covar_module = gpytorch.kernels.ScaleKernel(gpytorch.kernels.MaternKernel(nu=3/2, ard_num_dims=train_x.size(-1)))

    def forward(self, x):
        mean_x = self.mean_module(x)
        covar_x = self.covar_module(x)
        return gpytorch.distributions.MultivariateNormal(mean_x, covar_x)



#### Training & Prediction

In [ ]:
# Define row and column names
row_names = ['AUC', 'RMSE', 'ACC', 'LS','Time']
col_names = ['Iteration 1', 'Iteration 2', 'Iteration 3', 'Iteration 4', 'Iteration 5']

# Create an empty DataFrame with the specified row and column names
dat_SGPR = pd.DataFrame(np.zeros((len(row_names), len(col_names))), 
                      index=row_names, columns=col_names)

for j in range(5):

    # Data
    X_test = np.array([X[idx] for idx in test_indices_sets[j]])
    X_train = np.array([X[idx] for idx in train_indices_sets[j]])
    y_test = np.array([y[idx] for idx in test_indices_sets[j]])
    y_train = np.array([y[idx] for idx in train_indices_sets[j]])
    X_train, X_test, y_train, y_test = Scale(X_train, X_test, y_train, y_test)
    print(X_train.shape)
    print(y_train.shape)
    
    # Convert the numpy arrays to PyTorch tensors
    X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
    X_test_tensor = torch.tensor(X_test, dtype=torch.float32)
    y_train_tensor = torch.tensor(y_train, dtype=torch.float32)
    y_test_tensor = torch.tensor(y_test, dtype=torch.float32)

    mask_all = find_unique_X(X_train_tensor)
    X_train_tensor = X_train_tensor[mask_all]
    y_train_tensor = y_train_tensor[mask_all]
    print(X_train.shape)
    print(y_train.shape)
    
    print(f"Split {j+1} Test Response: {y_test}")
    
    # Model
    likelihood = gpytorch.likelihoods.BernoulliLikelihood()
    model_SGPR = SGPR(X_train_tensor, num_inducing_points)
    # Use the adam optimizer
    #optimizer = torch.optim.LBFGS(model_SGPR.parameters(), lr=0.1)
    optimizer = torch.optim.Adam(model_SGPR.parameters(), lr=0.1)
    if torch.cuda.is_available():
        model_SGPR = model_SGPR.cuda()
        likelihood = likelihood.cuda()
    # "Loss" for GPs - the marginal log likelihood
    mll = gpytorch.mlls.VariationalELBO(likelihood, model_SGPR,num_data=len(y_train_tensor))
    
    # Train
    def train():
        #iterator = tqdm.tqdm(range(training_iterations), desc="Train")
        print("Training started.")
        for i in range(training_iterations):
            def closure():
                optimizer.zero_grad()
                output = model_SGPR(X_train_tensor)
                loss = -mll(output, y_train_tensor)
                loss.backward()
                return loss

            loss = optimizer.step(closure)
        
            if (i+1) % 10 == 0:
                print(f"Iteration {i+1}/{training_iterations}")
                print(f"Loss: {loss.item()}\n")

        print("Training completed.")
    
    start_time = time.time()
    %time train()
    runtime_SGPR = time.time() - start_time

    # Prediction
    print("Prediction started.")
    model_SGPR.eval()
    likelihood.eval()
    with torch.no_grad(), gpytorch.settings.fast_pred_var():
        preds_SGPR = likelihood(model_SGPR(X_test_tensor))
        variances_SGPR = preds_SGPR.variance
        predicted_labels = (preds_SGPR.mean >= 0.5).float()  # Threshold at 0.5
        # Compute accuracy
        correct_predictions = (predicted_labels == y_test_tensor).float().sum()
        accuracy_SGPR = correct_predictions / y_test_tensor.size(0)
        auc_score_SGPR = roc_auc_score(y_test_tensor.numpy(), preds_SGPR.mean.numpy())
        MAE_SGPR = torch.mean(torch.abs(preds_SGPR.mean - y_test_tensor))
        RMSE_SGPR = torch.sqrt(torch.mean(torch.square(preds_SGPR.mean - y_test_tensor)))
        LS_SGPR = log_score_norm_vectorized(y_test_tensor,preds_SGPR.mean,variances_SGPR)
        CRPS_SGPR = crps_norm_vectorized(y_test_tensor,preds_SGPR.mean,variances_SGPR)
        dat_SGPR.loc[row_names[0], col_names[j]] = auc_score_SGPR.item()
        dat_SGPR.loc[row_names[1], col_names[j]] = RMSE_SGPR.item()
        dat_SGPR.loc[row_names[2], col_names[j]] = accuracy_SGPR.item()
        dat_SGPR.loc[row_names[3], col_names[j]] = LS_SGPR.item()
        dat_SGPR.loc[row_names[4], col_names[j]] = runtime_SGPR
        print('Test AUC: {}'.format(auc_score_SGPR))
        print('Test RMSE: {}'.format(RMSE_SGPR))
        print('Test ACC: {}'.format(accuracy_SGPR))
        print('Test LS: {}'.format(LS_SGPR))
        print('Runtime: {} seconds'.format(runtime_SGPR))
    

### Stochastic Variational Gaussian Processes (SVGP)

#### Parameters

In [ ]:
num_inducing_points = 1000
training_iterations = 500

#### Model

In [ ]:
class SVGP(ApproximateGP):
    def __init__(self, inducing_points):
        variational_distribution = CholeskyVariationalDistribution(inducing_points.size(0))
        variational_strategy = VariationalStrategy(self, inducing_points, variational_distribution, learn_inducing_locations=True)
        super(SVGP, self).__init__(variational_strategy)
        self.mean_module = gpytorch.means.ConstantMean()
        self.covar_module = gpytorch.kernels.ScaleKernel(MaternKernel(nu=3/2,ard_num_dims = d))

    def forward(self, x):
        mean_x = self.mean_module(x)
        covar_x = self.covar_module(x)
        return gpytorch.distributions.MultivariateNormal(mean_x, covar_x)

#### Training & Prediction

In [ ]:
# Define row and column names
row_names = ['AUC', 'RMSE', 'ACC', 'LS','Time']
col_names = ['Iteration 1', 'Iteration 2', 'Iteration 3', 'Iteration 4', 'Iteration 5']

# Create an empty DataFrame with the specified row and column names
dat_SVGP = pd.DataFrame(np.zeros((len(row_names), len(col_names))), 
                      index=row_names, columns=col_names)

for j in range(5):

    # Data
    X_test = np.array([X[idx] for idx in test_indices_sets[j]])
    X_train = np.array([X[idx] for idx in train_indices_sets[j]])
    y_test = np.array([y[idx] for idx in test_indices_sets[j]])
    y_train = np.array([y[idx] for idx in train_indices_sets[j]])
    X_train, X_test, y_train, y_test = Scale(X_train, X_test, y_train, y_test)
    print(X_train.shape)
    print(y_train.shape)
    
    # Convert the numpy arrays to PyTorch tensors
    X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
    X_test_tensor = torch.tensor(X_test, dtype=torch.float32)
    y_train_tensor = torch.tensor(y_train, dtype=torch.float32)
    y_test_tensor = torch.tensor(y_test, dtype=torch.float32)
    # Remove close points
    mask_all = find_unique_X(X_train_tensor)
    X_train_tensor = X_train_tensor[mask_all]
    y_train_tensor = y_train_tensor[mask_all]
    print(X_train.shape)
    print(y_train.shape)
    # Batches
    train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
    train_loader = DataLoader(train_dataset, batch_size=1024, shuffle=True)

    test_dataset = TensorDataset(X_test_tensor, y_test_tensor)
    test_loader = DataLoader(test_dataset, batch_size=1024, shuffle=False)
    
    print(f"Split {j+1} Test Response: {y_test}")
    
    # Model
    likelihood = gpytorch.likelihoods.BernoulliLikelihood()
    inducing_points = X_train_tensor[:num_inducing_points, :]
    model_SVGP = SVGP(inducing_points=inducing_points)
    if torch.cuda.is_available():
        model_SVGP = model_SVGP.cuda()
        likelihood = likelihood.cuda()
    # Use the adam optimizer
    model_SVGP.train()
    likelihood.train()
    optimizer = torch.optim.Adam([
        {'params': model_SVGP.parameters()},
        {'params': likelihood.parameters()},
    ], lr=0.01)
    # Our loss object. We're using the VariationalELBO
    mll = gpytorch.mlls.VariationalELBO(likelihood, model_SVGP, num_data=y_train_tensor.size(0))
    
    # Train
    def train():
        print("Training started.")
        #epochs_iter = tqdm.tqdm(range(training_iterations), desc="Epoch")
        for i in range(training_iterations):
            # Within each iteration, we will go over each minibatch of data
            #minibatch_iter = tqdm.tqdm(train_loader, desc="Minibatch", leave=False)
            for x_batch, y_batch in train_loader:
                optimizer.zero_grad()
                output = model_SVGP(x_batch)
                loss = -mll(output, y_batch)
                #minibatch_iter.set_postfix(loss=loss.item())
                loss.backward()
                optimizer.step()
            if (i+1) % 10 == 0:
                print(f"Iteration {i + 1}/{training_iterations}")
                print(f"Loss: {loss.item()}\n")
                
        print("Training completed.")
    
    start_time = time.time()
    %time train()
    runtime_SVGP = time.time() - start_time

    # Prediction
    print("Prediction started.")
    model_SVGP.eval()
    likelihood.eval()
    means_SVGP = torch.tensor([0.])
    variances_SVGP = torch.tensor([0.])
    with torch.no_grad():
        for x_batch, y_batch in test_loader:
            preds = likelihood(model_SVGP(x_batch))
            means_SVGP = torch.cat([means_SVGP, preds.mean.cpu()])
            variances_SVGP = torch.cat([variances_SVGP, preds.variance.cpu()])
        means_SVGP = means_SVGP[1:]
        variances_SVGP = variances_SVGP[1:]
        predicted_labels = (means_SVGP >= 0.5).float()  # Threshold at 0.5
        # Compute accuracy
        correct_predictions = (predicted_labels == y_test_tensor).float().sum()
        accuracy_SVGP = correct_predictions / y_test_tensor.size(0)
        auc_score_SVGP = roc_auc_score(y_test_tensor.numpy(), means_SVGP.numpy())
        MAE_SVGP = torch.mean(torch.abs(means_SVGP - y_test_tensor))
        RMSE_SVGP = torch.sqrt(torch.mean(torch.square(means_SVGP - y_test_tensor)))
        LS_SVGP = log_score_norm_vectorized(y_test_tensor,means_SVGP,variances_SVGP)
        CRPS_SVGP = crps_norm_vectorized(y_test_tensor,means_SVGP,variances_SVGP)
        dat_SVGP.loc[row_names[0], col_names[j]] = auc_score_SVGP.item()
        dat_SVGP.loc[row_names[1], col_names[j]] = RMSE_SVGP.item()
        dat_SVGP.loc[row_names[2], col_names[j]] = accuracy_SVGP.item()
        dat_SVGP.loc[row_names[3], col_names[j]] = LS_SVGP.item()
        dat_SVGP.loc[row_names[4], col_names[j]] = runtime_SVGP
        print('Test AUC: {}'.format(auc_score_SVGP))
        print('Test RMSE: {}'.format(RMSE_SVGP))
        print('Test ACC: {}'.format(accuracy_SVGP))
        print('Test LS: {}'.format(LS_SVGP))
        print('Runtime: {} seconds'.format(runtime_SVGP))

### Double-Kullback-Leibler-optimal Gaussian-process approximation (DKL-GP)

#### Parameters

In [ ]:
classify = True
rho = 1.5
n_Epoch=35
from viva import LogitLikelihood

#### Train & Prediction

In [ ]:
# Define row and column names
row_names = ['AUC', 'RMSE', 'ACC', 'LS','Time']
col_names = ['Iteration 1', 'Iteration 2', 'Iteration 3', 'Iteration 4', 'Iteration 5']

# Create an empty DataFrame with the specified row and column names
dat_DKL = pd.DataFrame(np.zeros((len(row_names), len(col_names))), 
                      index=row_names, columns=col_names)

for j in range(5):

    # Data
    X_test = np.array([X[idx] for idx in test_indices_sets[j]])
    X_train = np.array([X[idx] for idx in train_indices_sets[j]])
    y_test = np.array([y[idx] for idx in test_indices_sets[j]])
    y_train = np.array([y[idx] for idx in train_indices_sets[j]])
    X_train, X_test, y_train, y_test = Scale(X_train, X_test, y_train, y_test)
    print(X_train.shape)
    print(y_train.shape)
    
    # Convert the numpy arrays to PyTorch tensors
    X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
    X_test_tensor = torch.tensor(X_test, dtype=torch.float32)
    y_train_tensor = torch.tensor(y_train, dtype=torch.float32)
    y_test_tensor = torch.tensor(y_test, dtype=torch.float32)
    # Remove close points
    mask_all = find_unique_X(X_train_tensor)
    X_train_tensor = X_train_tensor[mask_all]
    y_train_tensor = y_train_tensor[mask_all]
    print(X_train.shape)
    print(y_train.shape)
    X_tensor = torch.cat((X_train_tensor, X_test_tensor), dim=0)
    y_tensor = torch.cat((y_train_tensor, y_test_tensor), dim=0)
    print(f"Split {j+1} Test Response: {y_test}")
    
    # Model
    likelihood = LogitLikelihood()
    # Kernel
    K = gpytorch.kernels.ScaleKernel(MaternKernel(nu=3/2,ard_num_dims = d))

    model_DKL = VIVA(X_tensor, y_tensor, K, likelihood, rho, n_test=y_test.size,
                                 classify=classify, use_ic0=True)
    
    start_time = time.time()
    my_train(model_DKL, n_Epoch=n_Epoch)
    runtime_DKL = time.time() - start_time

    # Prediction
    print("Prediction started.")
    preds_DKL, variances_DKL = model_DKL.predict()
    preds_DKL = 1.0 / (1.0 + (- preds_DKL).exp()).detach()
    predicted_labels = (preds_DKL >= 0.5).float()  # Threshold at 0.5
    # Compute accuracy
    correct_predictions = (predicted_labels == y_test_tensor).float().sum()
    accuracy_DKL = correct_predictions / y_test_tensor.size(0)
    auc_score_DKL = roc_auc_score(y_test_tensor.numpy(), preds_DKL.numpy())
    MAE_DKL = torch.mean(torch.abs(preds_DKL - y_test_tensor))
    RMSE_DKL = torch.sqrt(torch.mean(torch.square(preds_DKL - y_test_tensor)))
    CRPS_DKL = crps_norm_vectorized(y_test_tensor,preds_DKL,variances_DKL)
    LS_DKL = log_score_norm_vectorized(y_test_tensor,preds_DKL,variances_DKL)
    dat_DKL.loc[row_names[0], col_names[j]] = auc_score_DKL.item()
    dat_DKL.loc[row_names[1], col_names[j]] = RMSE_DKL.item()
    dat_DKL.loc[row_names[2], col_names[j]] = accuracy_DKL.item()
    dat_DKL.loc[row_names[3], col_names[j]] = LS_DKL.item()
    dat_DKL.loc[row_names[4], col_names[j]] = runtime_DKL
    print('Test AUC: {}'.format(auc_score_DKL))
    print('Test RMSE: {}'.format(RMSE_DKL))
    print('Test ACC: {}'.format(accuracy_DKL))
    print('Test LS: {}'.format(LS_DKL))
    print('Runtime: {} seconds'.format(runtime_DKL))



### VIF Approximation

#### Parameters

In [ ]:
likelihood = "bernoulli_logit"
num_inducing_points= 200
num_Vecchia_neighbors = 30

#### Training & Prediction

In [ ]:
# Define row and column names
row_names = ['AUC', 'RMSE', 'ACC', 'LS','Time']
col_names = ['Iteration 1', 'Iteration 2', 'Iteration 3', 'Iteration 4', 'Iteration 5']

# Create an empty DataFrame with the specified row and column names
dat_vif_corr = pd.DataFrame(np.zeros((len(row_names), len(col_names))), 
                      index=row_names, columns=col_names)
dat_vif_eucl = pd.DataFrame(np.zeros((len(row_names), len(col_names))), 
                      index=row_names, columns=col_names)

for j in range(5):
    X_test = np.array([X[idx] for idx in test_indices_sets[j]])
    X_train = np.array([X[idx] for idx in train_indices_sets[j]])
    y_test = np.array([y[idx] for idx in test_indices_sets[j]])
    y_train = np.array([y[idx] for idx in train_indices_sets[j]])
    X_train, X_test, y_train, y_test = Scale(X_train, X_test, y_train, y_test)
    print(X_train.shape)
    print(y_train.shape)
    
    # Convert the numpy arrays to PyTorch tensors
    X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
    y_train_tensor = torch.tensor(y_train, dtype=torch.float32)
    # Remove close points
    mask_all = find_unique_X(X_train_tensor)
    X_train_tensor = X_train_tensor[mask_all]
    y_train_tensor = y_train_tensor[mask_all]
    X_train = np.asarray(X_train_tensor)
    y_train = np.asarray(y_train_tensor)
    print(X_train.shape)
    print(y_train.shape)
    # correlation based Vecchia neighbor search
    model_vif_corr = gpb.GPModel(gp_coords=X_train, cov_function="matern_ard", cov_fct_shape=1.5,
                       likelihood=likelihood,vecchia_ordering = "random",num_neighbors =num_Vecchia_neighbors,
                       num_ind_points = num_inducing_points,ind_points_selection = "kmeans++",       
                       matrix_inversion_method = "iterative", gp_approx="full_scale_vecchia_correlation_based")
    # Euclidean based Vecchia neighbor search
    model_vif_eucl = gpb.GPModel(gp_coords=X_train, cov_function="matern_ard", cov_fct_shape=1.5,
                       likelihood=likelihood,vecchia_ordering = "random",num_neighbors =num_Vecchia_neighbors,
                       num_ind_points = num_inducing_points,ind_points_selection = "kmeans++",       
                       matrix_inversion_method = "iterative", gp_approx="full_scale_vecchia")

    model_vif_corr.set_optim_params(params={"optimizer_cov": "lbfgs", "trace": True, 
                                             "cg_preconditioner_type": "predictive_process_plus_diagonal",
                                             "piv_chol_rank": 200})
    start_time = time.time()
    model_vif_corr.fit(y=y_train);
    runtime_vif_corr = time.time() - start_time

    model_vif_eucl.set_optim_params(params={"optimizer_cov": "lbfgs", "trace": True, 
                                             "cg_preconditioner_type": "predictive_process_plus_diagonal",
                                             "piv_chol_rank": 200})
    start_time = time.time()
    model_vif_eucl.fit(y=y_train);
    runtime_vif_eucl = time.time() - start_time
    
    pred_vif_corr = model_vif_corr.predict(gp_coords_pred=X_test, predict_var=True)
    predicted_labels = (torch.tensor(pred_vif_corr['mu'], dtype=torch.float32) >= 0.5).float()  # Threshold at 0.5
    # Compute accuracy
    correct_predictions = (predicted_labels == y_test).float().sum()
    accuracy_vif_corr = correct_predictions / len(y_test)
    auc_score_vif_corr = roc_auc_score(y_test, pred_vif_corr['mu'])
    MAE_vif_corr = np.mean(np.abs(pred_vif_corr['mu'] - y_test))
    RMSE_vif_corr = np.sqrt(np.mean(np.square(pred_vif_corr['mu'] - y_test)))
    CRPS_vif_corr = crps_norm_vectorized(y_test,pred_vif_corr['mu'],pred_vif_corr['var'])
    LS_vif_corr = log_score_norm_vectorized(y_test,pred_vif_corr['mu'],pred_vif_corr['var'])
    print('Test AUC: {}'.format(auc_score_vif_corr))
    print('Test RMSE: {}'.format(RMSE_vif_corr))
    print('Test ACC: {}'.format(accuracy_vif_corr))
    print('Test LS: {}'.format(LS_vif_corr))
    print('Runtime: {} seconds'.format(runtime_vif_corr))
    dat_vif_corr.loc[row_names[0], col_names[j]] = auc_score_vif_corr.item()
    dat_vif_corr.loc[row_names[1], col_names[j]] = RMSE_vif_corr.item()
    dat_vif_corr.loc[row_names[2], col_names[j]] = accuracy_vif_corr.item()
    dat_vif_corr.loc[row_names[3], col_names[j]] = LS_vif_corr.item()
    dat_vif_corr.loc[row_names[4], col_names[j]] = runtime_vif_corr
    
    pred_vif_eucl = model_vif_eucl.predict(gp_coords_pred=X_test, predict_var=True)
    predicted_labels = (torch.tensor(pred_vif_eucl['mu'], dtype=torch.float32) >= 0.5).float()  # Threshold at 0.5
    # Compute accuracy
    correct_predictions = (predicted_labels == y_test).float().sum()
    accuracy_vif_eucl = correct_predictions / len(y_test)
    auc_score_vif_eucl = roc_auc_score(y_test, pred_vif_eucl['mu'])
    MAE_vif_eucl = np.mean(np.abs(pred_vif_eucl['mu'] - y_test))
    RMSE_vif_eucl = np.sqrt(np.mean(np.square(pred_vif_eucl['mu'] - y_test)))
    CRPS_vif_eucl = crps_norm_vectorized(y_test,pred_vif_eucl['mu'],pred_vif_eucl['var'])
    LS_vif_eucl = log_score_norm_vectorized(y_test,pred_vif_eucl['mu'],pred_vif_eucl['var'])
    print('Test AUC: {}'.format(auc_score_vif_eucl))
    print('Test RMSE: {}'.format(RMSE_vif_eucl))
    print('Test ACC: {}'.format(accuracy_vif_eucl))
    print('Test LS: {}'.format(LS_vif_eucl))
    print('Runtime: {} seconds'.format(runtime_vif_eucl))
    dat_vif_eucl.loc[row_names[0], col_names[j]] = auc_score_vif_eucl.item()
    dat_vif_eucl.loc[row_names[1], col_names[j]] = RMSE_vif_eucl.item()
    dat_vif_eucl.loc[row_names[2], col_names[j]] = accuracy_vif_eucl.item()
    dat_vif_eucl.loc[row_names[3], col_names[j]] = LS_vif_eucl.item()
    dat_vif_eucl.loc[row_names[4], col_names[j]] = runtime_vif_eucl
